In [ ]:
import os
import json
import random
import pandas as pd
import numpy as np
import scipy as sp
import pylab as plt
import seaborn as sns
from tqdm import tqdm

In [ ]:
%load_ext autoreload
%autoreload 2
import src.count_utils as utils

In [ ]:
plt.style.use("../src/mpl_style.txt")

In [ ]:
import jupyter_black
jupyter_black.load()

In [ ]:
BASELINE_NAME = "baseline_2025-06-26"
DATA_PATH = os.path.join("../data", BASELINE_NAME, "formatted")
sections = ["introduction", "methods", "results", "discussion"]

In [ ]:
df, c1, c2 = utils.load_paper_df(DATA_PATH, "research-article", sections, subset="all")
len(df)

In [ ]:
df[df["date"].apply(lambda x: x.year >= 2010)]

In [ ]:
journal_counts = df["journal"].value_counts()
disc_papers = np.sum(journal_counts[journal_counts < 100])
disc_journals = np.sum(journal_counts < 100)
journal_counts_agg = journal_counts[journal_counts >= 100]
journal_counts_agg.loc["other"] = disc_papers
journal_counts_agg

In [ ]:
fig, ax = plt.subplots(figsize=[6, 4])

plt.pie(
    journal_counts_agg,
    labels=journal_counts_agg.index,
    autopct="%.1f%%",
    pctdistance=0.9,
)
plt.title("Journal distribution 2010")
plt.text(
    x=-2,
    y=1,
    s=f"# papers before filters: {c1}\n# papers after filters: {c2}\n# journals: {len(journal_counts)}",
)

In [ ]:
"""
journal_counts_yearly = {}

for year in tqdm(range(2010, 2026)):
    df, c1, c2 = utils.load_paper_df(
        DATA_PATH, "research-article", sections, subset=year
    )
    journal_counts_yearly[str(year)] = {
        "journals": df["journal"].value_counts().to_dict(),
        "c1": c1,
        "c2": c2,
    }
"""

with open(os.path.join(DATA_PATH, "journal_counts.json"), "r") as f:
    journal_counts_yearly = json.load(f)

In [ ]:
all_journals = []

for year in range(2010, 2026):
    j = journal_counts_yearly[str(year)]["journals"].keys()
    all_journals += j

random.seed(0)
all_journals = list(set(all_journals))
random.shuffle(all_journals)
journals_colormap = dict(
    zip(
        all_journals,
        [cs for cs in sns.color_palette("hls", len(all_journals))],
    )
)

In [ ]:
yearly_additions = {}
yearly_big_additions = {}

for year in range(2011, 2026):
    js = journal_counts_yearly[str(year)]["journals"].keys()
    js_prev = journal_counts_yearly[str(year - 1)]["journals"].keys()
    js_new = [j for j in js if not j in js_prev]
    yearly_additions[str(year)] = js_new
    yearly_big_additions[str(year)] = [
        j for j in js_new if journal_counts_yearly[str(year)]["journals"][j] > 1
    ]

In [ ]:
yearly_removals = {}
yearly_big_removals = {}

for year in range(2011, 2026):
    js = journal_counts_yearly[str(year)]["journals"].keys()
    js_prev = journal_counts_yearly[str(year - 1)]["journals"].keys()
    js_new = [j for j in js_prev if not j in js]
    yearly_removals[str(year)] = js_new
    yearly_big_removals[str(year)] = [
        j for j in js_new if journal_counts_yearly[str(year - 1)]["journals"][j] > 1
    ]

In [ ]:
len(yearly_additions["2011"]) - len(yearly_removals["2011"])

In [ ]:
list(map(len, yearly_additions.values()))

In [ ]:
year = 2025
k = 10

print(f"{year}\nBiggest added journals:")
for i in range(k):
    j = yearly_additions[str(year)][i]
    print(f"{j} ({journal_counts_yearly[str(year)]["journals"][j]})")
print("\nBiggest removed journals:")
for i in range(k):
    j = yearly_removals[str(year)][i]
    print(f"{j} ({journal_counts_yearly[str(year-1)]["journals"][j]})")

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4), layout="constrained")


ax.plot(
    range(2011, 2026),
    list(map(len, yearly_additions.values())),
    marker="o",
    color="green",
    alpha=0.4,
    label="all added journals",
)
ax.plot(
    range(2011, 2026),
    list(map(len, yearly_big_additions.values())),
    marker="o",
    color="green",
    label="added journals with > 1 papers",
)
ax.plot(
    range(2011, 2026),
    list(map(lambda x: -len(x), yearly_removals.values())),
    marker="o",
    color="darkorange",
    alpha=0.4,
    label="all removed journals",
)
ax.plot(
    range(2011, 2026),
    list(map(lambda x: -len(x), yearly_big_removals.values())),
    marker="o",
    color="darkorange",
    label="removed journals with > 1 papers",
)
ax.legend(loc="upper left")
ax.spines["bottom"].set_position("center")
ax.set_ylim(-1500, 1500)
plt.title("Changes in Journal composition of PMC (w.r.t. the previous year)")

plt.savefig(os.path.join("../results/plots", "journal_deltas.png"), dpi=300)

In [ ]:
fig, axs = plt.subplots(nrows=4, ncols=4, figsize=(19, 12), layout="constrained")

for i, year in enumerate(range(2010, 2026)):

    journal_counts, c1, c2 = journal_counts_yearly[str(year)].values()
    journal_counts = pd.Series(journal_counts)

    # only show journals that make up >= 1% of papers
    thr = c2 * 0.01
    disc_papers = np.sum(journal_counts[journal_counts < thr])
    disc_journals = np.sum(journal_counts < thr)
    journal_counts_agg = journal_counts[journal_counts >= thr]

    other_label = f"other ({disc_journals} journals)"
    journal_counts_agg.loc[other_label] = disc_papers
    journals_colormap[other_label] = (0.5, 0.5, 0.5)

    # determine angle of smallest fraction
    i_min = np.argmin(journal_counts_agg)
    angles = 360 * journal_counts_agg / np.sum(journal_counts_agg)
    center_angle = angles[:i_min].sum() + angles[i_min] / 2
    start_angle = -center_angle

    ax = axs.flat[i]
    ax.pie(
        journal_counts_agg,
        labels=journal_counts_agg.index,
        autopct="%.1f%%",
        pctdistance=0.9,
        colors=[journals_colormap[key] for key in journal_counts_agg.index],
        startangle=start_angle,
    )

    ax.set_title(f"Journal distribution {year}")
    ax.text(
        x=1,
        y=0.9,
        s=f"# papers before filters: {c1}\n# papers after filters: {c2}\n# journals: {len(journal_counts)}",
    )